# 01 · Data Understanding
**Project:** Saudi Tourism & Hospitality Establishments Analytics – Q2 2026

Goal of this notebook: understand the seven cleaned datasets (shape, grain, keys, data types, units, missing values, duplicates), document them in a data dictionary, and export tidy tables to `data/processed/` for EDA and Power BI.

In [1]:
import pandas as pd
from IPython.display import Markdown, display
import prep_lib as prep
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 200)
main = prep.load_main()
xl = pd.ExcelFile(prep.RAW)
print('Sheets in workbook:', len(xl.sheet_names))
print('Analysis-ready (combined) sheets:', prep.MAIN)
print('Other sheets are the original numbered bulletin tables (1_x, 2_x, 3_x) that the 7 sheets were assembled from.')

Sheets in workbook: 55
Analysis-ready (combined) sheets: ['Quarterly_Trends', 'Regional_KPIs', 'City_KPIs', 'Employment_Activity', 'Employment_Region', 'Establishments_Activity', 'Establishments_Region']
Other sheets are the original numbered bulletin tables (1_x, 2_x, 3_x) that the 7 sheets were assembled from.


## 1. Shape, key, missing values and duplicates

In [2]:
keys = {'Quarterly_Trends':'Quarter','Regional_KPIs':'Region','City_KPIs':'City','Employment_Activity':'Activity','Employment_Region':'Region','Establishments_Activity':'Activity','Establishments_Region':'Region'}
grain = {'Quarterly_Trends':'1 row per quarter (Q1 2022 – Q2 2026), national level','Regional_KPIs':'1 row per administrative region, Q2 2026','City_KPIs':'1 row per city / city group, Q2 2026','Employment_Activity':'1 row per tourism activity, Q2 2026','Employment_Region':'1 row per region, Q2 2026','Establishments_Activity':'1 row per tourism activity, Q2 2026','Establishments_Region':'1 row per region, Q2 2026'}
rows = []
for s, d in main.items():
    k = keys[s]
    rows.append({'Dataset': s, 'Rows': len(d), 'Columns': d.shape[1], 'Key': k, 'Key is unique': d[k].is_unique,
                 'Missing cells': int(d.isna().sum().sum()), 'Duplicate rows': int(d.duplicated().sum()), 'Grain': grain[s]})
profile = pd.DataFrame(rows); profile

,Dataset,Rows,Columns,Key,Key is unique,Missing cells,Duplicate rows,Grain
0,Quarterly_Trends,18,17,Quarter,True,64,0,"1 row per quarter (Q1 2022 – Q2 2026), nationa..."
1,Regional_KPIs,13,50,Region,True,0,0,"1 row per administrative region, Q2 2026"
2,City_KPIs,17,46,City,True,0,0,"1 row per city / city group, Q2 2026"
3,Employment_Activity,12,14,Activity,True,0,0,"1 row per tourism activity, Q2 2026"
4,Employment_Region,13,13,Region,True,0,0,"1 row per region, Q2 2026"
5,Establishments_Activity,12,4,Activity,True,0,0,"1 row per tourism activity, Q2 2026"
6,Establishments_Region,13,4,Region,True,0,0,"1 row per region, Q2 2026"


**Reading the profile**
- Every table has a single-column key that is unique, and there are no duplicate rows.
- Only `Quarterly_Trends` has missing cells. They are *structural*, not errors: each indicator starts being published in a different quarter (see next cell).
- The seven tables sit at **four different grains** (quarter, region, city, activity). They must not be joined to each other row-by-row.

In [3]:
q = main['Quarterly_Trends']
start = {c: q.loc[q[c].notna(), 'Quarter'].iloc[0] for c in q.columns if q[c].isna().any()}
pd.DataFrame({'First quarter with data': start, 'Missing quarters': q.isna().sum()[list(start)]})

,First quarter with data,Missing quarters
1_1_Number_of_serviced_apartments_and_other_hospitality_facilities,Q1 of 2024,8
1_1_Number_of_hotels,Q1 of 2024,8
1_1_Total_licensed_hospitality,Q1 of 2024,8
1_2_Occupancy_rate_in_serviced_apartments_and_other_hospitality_facilities,Q1 of 2023,4
1_2_Occupancy_rate_in_hotels,Q1 of 2023,4
1_3_Average_daily_rate_in_serviced_apartments_and_other_hospitality_facilities,Q1 of 2023,4
1_3_Average_daily_rate_in_hotels,Q1 of 2023,4
1_4_Average_length_of_stay_in_serviced_apartments_and_other_hospitality_facilities,Q1 of 2023,4
1_4_Average_length_of_stay_in_hotels,Q1 of 2023,4
1_5_Total_number_of_Saudi_employees,Q1 of 2023,4


## 2. Data types and units

In [4]:
for s, d in main.items():
    print(f'--- {s}'); print(d.dtypes.astype(str).value_counts().to_string())

--- Quarterly_Trends
float64    13
str         2
int64       2
--- Regional_KPIs
float64    42
int64       6
str         2
--- City_KPIs
float64    45
str         1
--- Employment_Activity
int64      9
float64    3
str        2
--- Employment_Region
int64      9
str        2
float64    2
--- Establishments_Activity
str        2
int64      1
float64    1
--- Establishments_Region
str        2
int64      1
float64    1


In [5]:
r = main['Regional_KPIs']
checks = {
 'Occupancy rates are fractions between 0 and 1': bool(r.filter(regex='^2_(3|4)_').stack().between(0,1).all()),
 'ADR values are positive (SAR per night)': bool((r.filter(regex='^2_(9|10)_').stack() > 0).all()),
 'Length of stay is positive (nights)': bool((r.filter(regex='^2_(15|16)_').stack() > 0).all()),
 'Regional facilities sum = national licensed facilities (Q2 2026)': int(r['2_1_Total_licensed_facilities'].sum()) == int(q['1_1_Total_licensed_hospitality'].iloc[-1]),
 'Employment by activity total = employment by region total': int(main['Employment_Activity']['Total_Total'].sum()) == int(main['Employment_Region']['All_Total'].sum()),
 'Employment total = national total employees (Q2 2026)': int(main['Employment_Region']['All_Total'].sum()) == int(q['1_5_Total_employees'].iloc[-1]),
 'Establishments by activity total = by region total': int(main['Establishments_Activity']['Number_of_tourism_establishments'].sum()) == int(main['Establishments_Region']['Number_of_tourism_establishments'].sum()),
 'Establishments total = national establishments (Q2 2026)': int(main['Establishments_Region']['Number_of_tourism_establishments'].sum()) == int(q['1_6_Number_of_tourism_etablishments'].iloc[-1]),
 'Change_rate columns = Q2 2026 minus Q2 2025 (percentage-point / absolute for occupancy)': bool(((r['2_21_Q2_of_2026']-r['2_21_Q2_of_2025']) - r['2_21_Change_rate']).abs().max() < 1e-9),
}
pd.Series(checks, name='Passed').to_frame()

,Passed
Occupancy rates are fractions between 0 and 1,True
ADR values are positive (SAR per night),True
Length of stay is positive (nights),True
Regional facilities sum = national licensed facilities (Q2 2026),True
Employment by activity total = employment by region total,True
Employment total = national total employees (Q2 2026),True
Establishments by activity total = by region total,True
Establishments total = national establishments (Q2 2026),True
Change_rate columns = Q2 2026 minus Q2 2025 (percentage-point / absolute for occupancy),True


**Units used in the project**

| Indicator | Unit | Note |
|---|---|---|
| Occupancy rate | fraction 0–1 (shown as %) | room occupancy |
| ADR – Average Daily Rate | SAR per occupied room-night | assumed SAR (official Saudi statistics) |
| Average length of stay | nights | |
| Facilities / rooms / establishments / employees | counts | |
| Shares | fraction 0–1 | |

`Change_rate` columns: for occupancy they are the **difference** between Q2 2026 and Q2 2025 (percentage points); for ADR and length of stay they are **relative** change (%). Verified below.

In [6]:
print('ADR change is relative:', bool(((r['2_24_Q2_of_2026']/r['2_24_Q2_of_2025']-1) - r['2_24_Change_rate']).abs().max() < 1e-6))
print('ALOS change is relative:', bool(((r['2_26_Q2_of_2026']/r['2_26_Q2_of_2025']-1) - r['2_26_Change_rate']).abs().max() < 1e-6))

ADR change is relative: True
ALOS change is relative: True


## 3. Data dictionary (important columns)
Column prefixes such as `2_3_` are the table numbers of the source bulletin.

In [7]:
dd = [
('Quarterly_Trends','Quarter','text','Quarter label, e.g. "Q2 of 2026" (key)'),
('Quarterly_Trends','1_1_*','count','Licensed hotels / serviced apartments & other facilities / total (from Q1 2024)'),
('Quarterly_Trends','1_2_*','rate 0-1','Occupancy rate – hotels and serviced apartments (from Q1 2023)'),
('Quarterly_Trends','1_3_*','SAR','Average daily rate – hotels and serviced apartments'),
('Quarterly_Trends','1_4_*','nights','Average length of stay – hotels and serviced apartments'),
('Quarterly_Trends','1_5_*','count / share','Saudi, non-Saudi and total employees; female share of employees'),
('Quarterly_Trends','1_6_Number_of_tourism_etablishments','count','Tourism establishments, all activities (from Q1 2022)'),
('Regional_KPIs','Region','text','13 administrative regions (key)'),
('Regional_KPIs','2_1_* / 2_2_*','count','Licensed facilities and available rooms by facility type'),
('Regional_KPIs','2_3_* / 2_4_*','rate 0-1','Occupancy: serviced apartments (2_3) / hotels (2_4), monthly + quarterly average'),
('Regional_KPIs','2_9_* / 2_10_*','SAR','ADR: serviced apartments (2_9) / hotels (2_10)'),
('Regional_KPIs','2_15_* / 2_16_*','nights','Length of stay: serviced apartments (2_15) / hotels (2_16)'),
('Regional_KPIs','2_21 … 2_26','mixed','Q2 2025 vs Q2 2026 comparison for occupancy, ADR, length of stay (apartments, hotels)'),
('City_KPIs','City','text','16 cities / city groups + "Other" (key)'),
('City_KPIs','2_5–2_8','rate 0-1','Occupancy: by type (2_5), all facilities monthly (2_6), apartments (2_7), hotels (2_8)'),
('City_KPIs','2_11–2_14','SAR','ADR, same layout'),
('City_KPIs','2_17–2_20','nights','Length of stay, same layout'),
('Employment_Activity','Activity','text','12 tourism activities (key)'),
('Employment_*','Saudi_* / NonSaudi_* / Total_*','count','Employees by nationality and gender'),
('Establishments_*','Number_of_tourism_establishments','count','Establishments by activity or region'),
]
data_dictionary = pd.DataFrame(dd, columns=['Dataset','Column','Unit','Description'])
data_dictionary.to_csv(prep.ROOT/'reports'/'data_dictionary.csv', index=False)
data_dictionary

,Dataset,Column,Unit,Description
0,Quarterly_Trends,Quarter,text,"Quarter label, e.g. ""Q2 of 2026"" (key)"
1,Quarterly_Trends,1_1_*,count,Licensed hotels / serviced apartments & other ...
2,Quarterly_Trends,1_2_*,rate 0-1,Occupancy rate – hotels and serviced apartment...
3,Quarterly_Trends,1_3_*,SAR,Average daily rate – hotels and serviced apart...
4,Quarterly_Trends,1_4_*,nights,Average length of stay – hotels and serviced a...
5,Quarterly_Trends,1_5_*,count / share,"Saudi, non-Saudi and total employees; female s..."
6,Quarterly_Trends,1_6_Number_of_tourism_etablishments,count,"Tourism establishments, all activities (from Q..."
7,Regional_KPIs,Region,text,13 administrative regions (key)
8,Regional_KPIs,2_1_* / 2_2_*,count,Licensed facilities and available rooms by fac...
9,Regional_KPIs,2_3_* / 2_4_*,rate 0-1,Occupancy: serviced apartments (2_3) / hotels ...


## 4. Tidy tables for analysis and Power BI
The wide bulletin layout (one column per month / facility type / gender) is reshaped into long **fact tables**, each kept at its own grain, plus small **dimension tables**.

In [8]:
tables = prep.build()
pd.DataFrame([{'Table': k, 'Rows': len(v), 'Columns': v.shape[1], 'Fields': ', '.join(v.columns)} for k, v in tables.items()])

,Table,Rows,Columns,Fields
0,Dim_Quarter,18,5,"Quarter_Key, Quarter_Label, Year, Quarter_Num,..."
1,Fact_Quarterly,18,15,"Apartments_Count, Hotels_Count, Licensed_Facil..."
2,Fact_Regional_Hospitality,26,10,"Region, Facility_Type, Facilities, Available_R..."
3,Fact_Regional_Monthly,78,7,"Region, Facility_Type, Month, Month_Num, Occup..."
4,Fact_City_Hospitality,51,5,"City, Facility_Type, Occupancy, ADR, ALOS"
5,Fact_City_Monthly,153,7,"City, Facility_Type, Month, Month_Num, Occupan..."
6,Fact_Employment_Activity,48,4,"Activity, Nationality, Gender, Employees"
7,Fact_Employment_Region,52,4,"Region, Nationality, Gender, Employees"
8,Fact_Establishments_Activity,12,2,"Activity, Establishments"
9,Fact_Establishments_Region,13,2,"Region, Establishments"


## 5. Limitations noted at this stage
1. **Single reference quarter** for regional, city, employment and establishment tables (Q2 2026). Only national indicators have a time series.
2. **Unequal history**: establishments from 2022, operating KPIs and employment from 2023, licensed facility counts from 2024.
3. **Averages cannot be re-aggregated naively.** Regional occupancy/ADR are already averages; a correct national figure needs weighting by rooms (done in notebook 02).
4. **City table is not a breakdown of the region table**: cities are selected cities/city groups plus an `Other` bucket, with no room counts, so they cannot be weighted or rolled up to regions.
5. Employment and establishments cover **all 12 tourism activities**, while occupancy/ADR/length of stay cover **accommodation only**.
6. Source footnotes for `Riyadh city (1)`, `Dammam metropolitan area (2)` and `Other (3)` are not included in the workbook.